# Rananiti Pressure Detection

Apply context-aware pressure labels and inspect how labels vary across innings and match stages.

All inputs are loaded from `data/raw/`. Outputs are saved to `outputs/`.

In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(min(8, os.cpu_count() or 2)))
ROOT_DIR=Path.cwd()
if not (ROOT_DIR/"data"/"raw"/"ball_by_ball.csv").exists(): ROOT_DIR=ROOT_DIR.parent
sys.path.insert(0,str(ROOT_DIR))
from config import BALLS_FILE, MATCHES_FILE, OUTPUTS
OUTPUTS.mkdir(parents=True,exist_ok=True)
balls=pd.read_csv(BALLS_FILE,low_memory=False)
matches=pd.read_csv(MATCHES_FILE,low_memory=False) if MATCHES_FILE.exists() else pd.DataFrame()
print("Project:",ROOT_DIR,"| ball rows:",len(balls),"| matches:",matches.match_id.nunique() if not matches.empty else "metadata missing")
from src.pressure_engine.match_state import reconstruct_match_state
from src.pressure_engine.pressure_detection import detect_pressure
state=detect_pressure(reconstruct_match_state(balls,matches if not matches.empty else None))
summary=state.groupby(["innings","pressure_level"],observed=False).size().rename("deliveries").reset_index()
print("Pressure index / label summary")
display(summary)
print("High-pressure rate:",round(100*state.is_pressure.mean(),2),"percent")
summary.to_csv(OUTPUTS/"pressure_summary.csv",index=False)
print("Saved:",OUTPUTS/"pressure_summary.csv")


Project: C:\Users\SHEETAL\OneDrive\ドキュメント\Desktop\Rananiti | ball rows: 200999 | matches: 883


Pressure index / label summary


,innings,pressure_level,deliveries
0,1,High,20066
1,1,Low,45172
2,1,Medium,40489
3,2,High,11521
4,2,Low,36794
5,2,Medium,46919
6,3,Low,25
7,4,Low,13


High-pressure rate: 15.8 percent
Saved: C:\Users\SHEETAL\OneDrive\ドキュメント\Desktop\Rananiti\outputs\pressure_summary.csv
